# Notebook #01 — Data Ingestion (Bronze Layer — 12 Canonical `FIELDS`)

### Mục tiêu & Nguyên tắc Bronze
1. Đọc toàn bộ dữ liệu gốc thô từ `data/raw/fids/`, `data/raw/aviationstack/`, `data/raw/weather/` (giữ nguyên dữ liệu gốc).
2. Kiểm tra schema hỗn hợp ban đầu (**58 cột thô** qua 11 file, tỷ lệ thiếu trường gate/baggage).
3. Chuẩn hóa ánh xạ về **12 `FIELDS` thống nhất** (`source, airport, direction, scraped_at, flight_date, airline_code, flight_no, other_airport, scheduled_time, estimated_time, actual_time, status`).
4. **Giữ nguyên trạng thái `status` gốc từ nguồn** (ví dụ *Departed 18 minutes late*, *Arrived On-Time*, *Cancelled*, *Scheduled*), **chưa** gom nhãn thành 3 lớp hay lọc bỏ ở Bronze.
5. Kiểm tra hợp đồng Bronze (`validate_raw_contract`) & lưu `data/interim/standardized_snapshots.csv` + `data/manifests/bronze_manifest.json`.

In [1]:
import json
from pathlib import Path
import pandas as pd

from flight_delay.common.constants import FIELDS, FILE_STANDARDIZED, RAW_AVIATIONSTACK_DIR, RAW_FIDS_DIR, RAW_WEATHER_DIR
from flight_delay.data_engineering.contracts import validate_raw_contract

fids_files = sorted(RAW_FIDS_DIR.glob("*.csv"))
av_files = sorted(RAW_AVIATIONSTACK_DIR.glob("*.csv"))
wx_file = RAW_WEATHER_DIR / "weather_hourly.csv"

print(f"FIDS raw files: {len(fids_files)} | Aviationstack files: {len(av_files)} | Weather cache exists: {wx_file.exists()}")
raw_cols = set()
total_raw_rows = 0
for p in av_files + fids_files:
    df_tmp = pd.read_csv(p, nrows=5)
    n_rows = sum(1 for _ in p.open("r", encoding="utf-8", errors="ignore")) - 1
    total_raw_rows += n_rows
    raw_cols.update(df_tmp.columns)
    print(f"  - {p.name:30s}: {n_rows:6,d} rows | {len(df_tmp.columns):2d} cols")

print(f"Total raw mixed rows: {total_raw_rows:,} | Unique raw column names: {len(raw_cols)}")

FIDS raw files: 10 | Aviationstack files: 1 | Weather cache exists: True
  - aviationstack_api_202610.csv  :  1,454 rows | 48 cols
  - guide_cxr_202610.csv          :  8,653 rows | 12 cols
  - guide_cxr_202611.csv          :    976 rows | 12 cols
  - guide_dad_202610.csv          : 12,168 rows | 12 cols
  - guide_dad_202611.csv          :  1,642 rows | 12 cols
  - guide_han_202610.csv          : 23,073 rows | 12 cols
  - guide_han_202611.csv          :  5,208 rows | 12 cols
  - guide_pqc_202610.csv          :  8,912 rows | 12 cols
  - guide_pqc_202611.csv          :  1,051 rows | 12 cols
  - guide_sgn_202610.csv          : 23,455 rows | 12 cols
  - guide_sgn_202611.csv          :  5,558 rows | 12 cols
Total raw mixed rows: 92,150 | Unique raw column names: 58


In [2]:
bronze_df = pd.read_csv(FILE_STANDARDIZED, dtype=str, keep_default_na=False)
ok_bronze, errs = validate_raw_contract(bronze_df)
assert ok_bronze, f"Bronze contract failed: {errs}"

print(f"Bronze standardized shape: {bronze_df.shape} | Canonical 12 FIELDS: {list(bronze_df.columns) == FIELDS}")
print("\nTop 10 raw status strings preserved before Silver normalization:")
print(bronze_df["status"].value_counts().head(10).to_string())
bronze_df[FIELDS].head(5)

Bronze standardized shape: (93081, 12) | Canonical 12 FIELDS: True

Top 10 raw status strings preserved before Silver normalization:
status
On-Time                     45156
Arrived On-Time             16722
Departed On-Time            16449
Cancelled                     483
Departed 15 minutes late      309
Arrived 15 minutes late       275
Departed 20 minutes late      271
Arrived 20 minutes late       259
Departed 16 minutes late      243
Arrived 18 minutes late       232
